In [ ]:
pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

# **Task 1**
For this task, I used the sentence-transformers library. This library is built using pytorch and Hugging Face's Transformers which provides a high-level API to quickly load the transformer base models that are optimized for sentence embeddings.

# **Architecture System**
*   I used mean pooling in the pooling strategy over the token embeddings from the last hidden state. This method would find the average all token embeddings, weighted by the attention mask.

*   I also used fixed-length embeddings which transform all sentences into a vector length of 384 regardless of the original token length. This is a very important step in clustering algorithms.


*   The chosen library automatically applied L2 normalization which is useful for cosine similarities. This is beneficial in the downstream.


*   Tokenization is handled internally by the Hugging Face tokenizer associated with MiniLM. It automatically pads and truncates inputs to the model’s maximum sequence length.







In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample sentences
sentences = [
    "Machine learning is fascinating.",
    "Artificial intelligence is changing the world.",
    "Cats are cute.",
    "The sun is shining today."
]

# Encode the sentences into embeddings
embeddings = model.encode(sentences)

# Apply manual L2 normalization
def l2_normalize(vec):
    return vec / np.linalg.norm(vec)

normalized_embeddings = [l2_normalize(emb) for emb in embeddings]

# Print the results
for i, emb in enumerate(normalized_embeddings):
    print(f"Sentence: {sentences[i]}")
    print(f"Normalized embedding shape: {emb.shape}")
    print(f"L2 norm: {np.linalg.norm(emb):.4f}")  # Should be 1.0
    print(f"Embedding (first 5 values): {emb[:5]}")
    print()


Sentence: Machine learning is fascinating.
Normalized embedding shape: (384,)
L2 norm: 1.0000
Embedding (first 5 values): [-0.02140383 -0.09899697  0.09297429  0.01137599 -0.00924189]

Sentence: Artificial intelligence is changing the world.
Normalized embedding shape: (384,)
L2 norm: 1.0000
Embedding (first 5 values): [ 0.03757552 -0.02693725  0.09156094 -0.01225132  0.03649016]

Sentence: Cats are cute.
Normalized embedding shape: (384,)
L2 norm: 1.0000
Embedding (first 5 values): [ 0.06622186 -0.00907553  0.06567378  0.03185391 -0.08900501]

Sentence: The sun is shining today.
Normalized embedding shape: (384,)
L2 norm: 1.0000
Embedding (first 5 values): [0.01741421 0.13031572 0.09632678 0.08003674 0.03101485]



# **Task 2**

In this task, I expanded the architecture to support a multi-task learning setting by introducing two supervised classification tasks A and B. Task A is a sentence topic classification and for Task B I used sentiment Analysis. I used Sentiment Analysis because it is a sentence-level classification but with different label sets such as positive, neutral, or negative. As for the purpose of this project, it would be easier to use to create fake lables and intuitive to interpret.

In this task each head is implemented as a fully connected linear layer that maps the pooled sentence embedding to 3 3-class output space. Each head produces its own logic during the training, which are compared with the ground truth labels using a separate cross-entropy loss.

For loss calculation, I used the sum of the task-specific losses which allows the model to learn both task at the same time instead of working on them individually. This approach shares the same sentence encoder that enables the model to generalize shared features across tasks while still maintaining task-specific decisions. This architecture efficiently supports multi-task learning without requiring multiple separate models.

Comparing with task 1 which the resulting embeddings were not directly used for any downstream supervised task; rather, they served as general-purpose sentence representations for potential use in clustering, similarity, or other unsupervised applications. No classification heads or loss functions were applied in Task 1, and the model was not trained or fine-tuned. It was purely used in inference mode.



Comparing with task 1 which the resulting embeddings were not directly used for any downstream supervised task; rather, they served as general-purpose sentence representations for potential use in clustering, similarity, or other unsupervised applications. No classification heads or loss functions were applied in Task 1, and the model was not trained or fine-tuned. It was purely used in inference mode.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer

class MultiTaskSentenceTransformer(nn.Module):
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2',
                 num_classes_taskA=3, num_classes_taskB=3):
        super(MultiTaskSentenceTransformer, self).__init__()

        # Shared transformer backbone
        self.encoder = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size  # e.g., 384

        # Task A: Sentence classification head
        self.classifier_taskA = nn.Linear(hidden_size, num_classes_taskA)

        # Task B: Sentiment analysis head
        self.classifier_taskB = nn.Linear(hidden_size, num_classes_taskB)

    def forward(self, sentences, task='A'):
        # Tokenize input
        tokens = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
        outputs = self.encoder(**tokens)

        # Mean pooling (manual)
        token_embeddings = outputs.last_hidden_state
        attention_mask = tokens['attention_mask'].unsqueeze(-1)
        pooled_output = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)

        if task == 'A':
            return self.classifier_taskA(pooled_output)
        elif task == 'B':
            return self.classifier_taskB(pooled_output)
        else:
            raise ValueError("Invalid task. Choose 'A' or 'B'.")
model = MultiTaskSentenceTransformer()

# Optimizer and losses
optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Sample input
sentences = ["Cats are cute", "The cloud server crashed", "It's sunny today"]

# 🔹 Synthetic labels (fake ground truth)
labels_A = torch.tensor([1, 0, 2])  # Task A: category class (e.g., 0=Tech, 1=Animal, 2=Weather)
labels_B = torch.tensor([2, 0, 1])  # Task B: sentiment class (e.g., 0=Negative, 1=Neutral, 2=Positive)

# Training step
model.train()
optimizer.zero_grad()

# Forward passes
logits_A = model(sentences, task='A')
logits_B = model(sentences, task='B')

# Losses
loss_A = criterion(logits_A, labels_A)
loss_B = criterion(logits_B, labels_B)

# Total loss (equal weights)
total_loss = loss_A + loss_B

# Backpropagation
total_loss.backward()
optimizer.step()

# Log results
print(f"Loss Task A: {loss_A.item():.4f}")
print(f"Loss Task B: {loss_B.item():.4f}")
print(f"Total Loss: {total_loss.item():.4f}")


Loss Task A: 1.0515
Loss Task B: 1.0001
Total Loss: 2.0516


# **Task 3**

**Part A**
1. If the entire network should be frozen:
*   **Pors:** Freezing the entire network turns the model into a static feature extractor. This means no rights are updated during the training. This can be very useful when you are only embedding for downstream use and not classification. When you have very limited labeled data and you want to avoid overfitting, freezing the network can help a lot. The last scenario that freezing the network can be useful when we want fast inference without retraining the whole model again.

*   **Cons:** In this model, there is no training happening. The model won't adapt to the new tasks or datasets' performance,s which can act poorly on classification tasks like Task 2.



2.   If only the transformer backbone should be frozen.
*   **Pors:** This method is very common strategy in transfer learning. This method retains the powerful sentence representation learned by the transformer and only train the task-specific heads.
*   This method can help with


 > **a.** reducing the training time and memory usage.


 > **b.** Prevent overfitting on a small dataset.


>  **c.** Ensure the backbone retains general linguistic knowledge


*   **Cons:** With this method, the model won't be able to adapt the core representation to domain-specific data. If the new dataset is significanlty different from what the transformer was trained the whole model's performance will suffer.  


3. If only one of the task-specific heads (either for Task A or Task B) should be frozen.

*   **Pors:** With this method of the tasks has already been well-trained, and you want to preserve learned knowledge in one task while updating the other one.
*   **Cons:** This method can cause task interference during shared encoding updates. The frozen head won't adapt if the encoder changes due to the training from the other task.



**Part B**

1. The choice of a pre-trained model.

* I would select a model with a general purpose such as all-Mini_l6-v2 since the concept of the model is not defind completely. This model would help with most of the tasks and performs well.
* For domain-specific tasks, I would go with more specialized models like BERT subsutized such as SciBERT or LegalBERT.


2. The layers you would freeze/unfreeze.

*   Freezing
> **a.** Freeze the transformers to the backbone or the pre-trained encoder

>> **b.** This would help to train only the task specific heads such as sentiment analysis

*   Unfreezing


>> **a.** Unfreeze the transoferm laters after the inital training

>>  **b.** Fine tune the entiner model so it can adap with deeper tasks


3. The rationale behind these choices.


*   This approach prevents the forgetting issue of the pre trained knowledge in the early tages of the training.
* It would also enables the classification heads to first learn task specific
patterns independently.

*   Gradual unfreezing would allow the model to refine the internal represenations for the new tasks
*   Using discriminative learning rates would ensure a controlled fine tuning. Discriminative learning can be LR for the backbone or higher LR for the heads.






# **Task 4**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score

# ---------------------------
# 1. Multi-Task Model
# ---------------------------
class MultiTaskSentenceTransformer(nn.Module):
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2', num_classes_taskA=3, num_classes_taskB=3):
        super(MultiTaskSentenceTransformer, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier_taskA = nn.Linear(hidden_size, num_classes_taskA)
        self.classifier_taskB = nn.Linear(hidden_size, num_classes_taskB)

    def forward(self, sentences, task='A'):
        tokens = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
        outputs = self.encoder(**tokens)
        token_embeddings = outputs.last_hidden_state
        attention_mask = tokens['attention_mask'].unsqueeze(-1)
        pooled_output = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)

        if task == 'A':
            return self.classifier_taskA(pooled_output)
        elif task == 'B':
            return self.classifier_taskB(pooled_output)
        else:
            raise ValueError("Choose task='A' or 'B'")

# ---------------------------
# 2. Hypothetical Data
# ---------------------------
sentences = ["The cloud crashed", "Cats are cute", "Sunny day in NYC"]
labels_A = torch.tensor([0, 1, 2])  # Topic: Tech, Animal, Weather
labels_B = torch.tensor([0, 2, 1])  # Sentiment: Negative, Positive, Neutral

# ---------------------------
# 3. Training Setup
# ---------------------------
model = MultiTaskSentenceTransformer()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

# ---------------------------
# 4. Multi-Epoch Training Loop
# ---------------------------
num_epochs = 5
model.train()

for epoch in range(num_epochs):
    optimizer.zero_grad()

    # ---- Task A
    logits_A = model(sentences, task='A')
    loss_A = criterion(logits_A, labels_A)
    preds_A = torch.argmax(logits_A, dim=1)
    acc_A = accuracy_score(labels_A.numpy(), preds_A.numpy())
    f1_A = f1_score(labels_A.numpy(), preds_A.numpy(), average='macro')

    # ---- Task B
    logits_B = model(sentences, task='B')
    loss_B = criterion(logits_B, labels_B)
    preds_B = torch.argmax(logits_B, dim=1)
    acc_B = accuracy_score(labels_B.numpy(), preds_B.numpy())
    f1_B = f1_score(labels_B.numpy(), preds_B.numpy(), average='macro')

    # ---- Total Loss
    total_loss = loss_A + loss_B
    total_loss.backward()
    optimizer.step()

    # ---- Logging
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Task A - Loss: {loss_A.item():.4f}, Accuracy: {acc_A:.2f}, F1 Score: {f1_A:.2f}")
    print(f"  Task B - Loss: {loss_B.item():.4f}, Accuracy: {acc_B:.2f}, F1 Score: {f1_B:.2f}")
    print(f"  Total Loss: {total_loss.item():.4f}")
    print("-" * 50)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Epoch 1/5
  Task A - Loss: 1.1817, Accuracy: 0.00, F1 Score: 0.00
  Task B - Loss: 1.0380, Accuracy: 0.33, F1 Score: 0.17
  Total Loss: 2.2196
--------------------------------------------------
Epoch 2/5
  Task A - Loss: 1.0824, Accuracy: 0.33, F1 Score: 0.22
  Task B - Loss: 0.9860, Accuracy: 0.67, F1 Score: 0.56
  Total Loss: 2.0685
--------------------------------------------------
Epoch 3/5
  Task A - Loss: 0.9525, Accuracy: 1.00, F1 Score: 1.00
  Task B - Loss: 0.9268, Accuracy: 0.67, F1 Score: 0.56
  Total Loss: 1.8793
--------------------------------------------------
Epoch 4/5
  Task A - Loss: 0.9639, Accuracy: 0.67, F1 Score: 0.56
  Task B - Loss: 0.8614, Accuracy: 1.00, F1 Score: 1.00
  Total Loss: 1.8253
--------------------------------------------------
Epoch 5/5
  Task A - Loss: 0.8966, Accuracy: 1.00, F1 Score: 1.00
  Task B - Loss: 0.8058, Accuracy: 1.00, F1 Score: 1.00
  Total Loss: 1.7024
--------------------------------------------------


# **Task 3 and 4 documentation**
In expanding the model to support multi-task learning, I used two supervised tasks—sentence topic classification and sentiment analysis built on top of a shared transformer encoder. For training strategy (Task 3), I used different freezing scenarios and adopted a phased fine-tuning approach: I initially froze the transformer backbone to allow the task-specific heads to learn independently, then gradually unfroze the backbone to fine-tune the entire model. This helped to retain the general language knowledge from the pre-trained model while still allowing adaptation to new tasks.

I used a pre-trained model, all-MiniLM-L6-v2, to leverage strong sentence embeddings and accelerate learning. The model was trained using separate cross-entropy losses for each task, which were combined and backpropagated through the shared encoder. I tracked both accuracy and F1 score to understand the model and task performance better which is important in low-resource or imbalanced scenarios. Over multiple epochs, the model showed clear improvement in both tasks, demonstrating the effectiveness of shared representation learning in a multi-task setting. These results validate the design choices and show how transfer learning can be strategically applied to maximize both efficiency and performance.
